<div style="direction: rtl; white-space: normal; line-height: 1;">
# Build Quran Embeddings

در این مرحله:

- دیتاست تمیزشده قرآن خوانده می‌شود.
- متن هر آیه برای embedding آماده می‌شود.
- مدل چندزبانه بارگذاری می‌شود.
- بردارهای آیات ساخته و ذخیره می‌شوند.
</div>

In [1]:
%pip install -q sentence-transformers numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
hazm 0.10.0 requires numpy==1.24.3, but you have numpy 2.2.6 which is incompatible.
gruut 2.2.3 requires numpy<2.0.0,>=1.19.0, but you have numpy 2.2.6 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import platform

print("Python:", sys.version)
print("Machine:", platform.machine())
print("Executable:", sys.executable)

Python: 3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]
Machine: x86_64
Executable: /usr/local/bin/python3


In [3]:
%pip install --force-reinstall "numpy==1.24.3"

  Using cached numpy-1.24.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.3-cp311-cp311-macosx_10_9_x86_64.whl (19.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Verify the environment after kernel restart

import numpy as np
import torch
import sentence_transformers

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("MPS available:", torch.backends.mps.is_available())

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 1.24.3
PyTorch: 2.2.2
Sentence Transformers: 5.0.0
MPS available: False


<div style="direction: rtl; white-space: normal; line-height: 1;">
فقط مدل را بارگذاری و با یک متن کوتاه تست می‌کنیم.
</div>

In [2]:
# Load embedding model

from sentence_transformers import SentenceTransformer

model_name = "intfloat/multilingual-e5-small"

model = SentenceTransformer(
    model_name,
    device="cpu"
)

test_text = ["passage: خداوند بخشنده و مهربان است"]

test_embedding = model.encode(
    test_text,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Model:", model_name)
print("Embedding shape:", test_embedding.shape)
print("Embedding dtype:", test_embedding.dtype)

Model: intfloat/multilingual-e5-small
Embedding shape: (1, 384)
Embedding dtype: float32


<div style="direction: rtl; white-space: normal; line-height: 1;">
ساخت متن ورودی هر آیه برای embedding
</div>

In [3]:
# Load cleaned Quran dataset and prepare passages

from pathlib import Path
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

input_file = project_root / "data" / "processed" / "quran_dataset_clean.json"

with open(input_file, "r", encoding="utf-8") as f:
    quran_data = json.load(f)

passages = []

for item in quran_data:
    passage = (
        f"passage: "
        f"سوره {item['surah']} آیه {item['ayah']} | "
        f"عربی: {item['arabic_clean']} | "
        f"ترجمه فولادوند: {item['fooladvand_clean']} | "
        f"ترجمه انصاریان: {item['ansarian_clean']}"
    )

    passages.append(passage)

print("Records:", len(quran_data))
print("Passages:", len(passages))
print("\nSample passage:")
print(passages[0])

Records: 6236
Passages: 6236

Sample passage:
passage: سوره 1 آیه 1 | عربی: بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِیمِ | ترجمه فولادوند: به نام خداوند رحمتگر مهربان | ترجمه انصاریان: به نام خدا که رحمتش بی‌اندازه است و مهربانی‌اش همیشگی.


<div style="direction: rtl; white-space: normal; line-height: 1;">
ساخت embedding برای هر ۶۲۳۶ آیه و ذخیره روی دیسک
</div>

In [4]:
# Generate and save Quran embeddings

import numpy as np
from pathlib import Path

embeddings_dir = project_root / "data" / "embeddings"
embeddings_dir.mkdir(parents=True, exist_ok=True)

embeddings_file = embeddings_dir / "quran_embeddings.npy"

quran_embeddings = model.encode(
    passages,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(embeddings_file, quran_embeddings)

print("Embeddings shape:", quran_embeddings.shape)
print("Embeddings dtype:", quran_embeddings.dtype)
print("Saved:", embeddings_file)

Batches: 100%|██████████| 390/390 [10:57<00:00,  1.69s/it]

Embeddings shape: (6236, 384)
Embeddings dtype: float32
Saved: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/embeddings/quran_embeddings.npy


<div style="direction: rtl; white-space: normal; line-height: 1;">
یک تست Retrieval بگیریم تا مطمئن شویم بردارها واقعاً آیات مرتبط را برمی‌گردانند.
</div>

In [5]:
# Test semantic retrieval

query = "خداوند مهربان و بخشنده است"

query_embedding = model.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

scores = quran_embeddings @ query_embedding[0]

top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

print("Query:", query)
print()

for rank, index in enumerate(top_indices, start=1):
    item = quran_data[index]

    print(f"Rank {rank}")
    print(f"Score: {scores[index]:.4f}")
    print(f"Surah: {item['surah']} | Ayah: {item['ayah']}")
    print("Arabic:", item["arabic"])
    print("Fooladvand:", item["fooladvand"])
    print("Ansarian:", item["ansarian"])
    print("-" * 80)

Query: خداوند مهربان و بخشنده است

Rank 1
Score: 0.9019
Surah: 3 | Ayah: 129
Arabic: وَلِلَّهِ مَا فِى ٱلسَّمَٰوَٰتِ وَمَا فِى ٱلْأَرْضِ يَغْفِرُ لِمَن يَشَآءُ وَيُعَذِّبُ مَن يَشَآءُ وَٱللَّهُ غَفُورٌ رَّحِيمٌ
Fooladvand: و آنچه در آسمانها و آنچه در زمین است از آنِ خداست. هر که را بخواهد می‌آمرزد، و هر که را بخواهد عذاب می‌کند، و خداوند، آمرزنده مهربان است.
Ansarian: و آنچه در آسمان ها و آنچه در زمین است، فقط در سیطره مالکیّت و فرمانروایی خداست. هر که را بخواهد می آمرزد و هر که را بخواهد عذاب می کند؛ و خدا بسیار آمرزنده و مهربان است.
--------------------------------------------------------------------------------
Rank 2
Score: 0.8981
Surah: 48 | Ayah: 14
Arabic: وَلِلَّهِ مُلْكُ ٱلسَّمَٰوَٰتِ وَٱلْأَرْضِ يَغْفِرُ لِمَن يَشَآءُ وَيُعَذِّبُ مَن يَشَآءُ وَكَانَ ٱللَّهُ غَفُورًا رَّحِيمًا
Fooladvand: و فرمانروایی آسمانها و زمین از آن خداست: هر که را بخواهد می‌بخشاید و هر که را بخواهد عذاب می‌کند، و خدا همواره آمرزنده مهربان است.
Ansarian: و مالکیّت و فرمانروایی آسمان ها و زمین فقط در 

<div style="direction: rtl; white-space: normal; line-height: 1;">
تست کنیم embedding ذخیره‌شده بعداً بدون ساخت دوباره قابل بارگذاری است.
</div>

In [6]:
# Reload and validate saved embeddings

loaded_embeddings = np.load(embeddings_file)

print("Loaded shape:", loaded_embeddings.shape)
print("Loaded dtype:", loaded_embeddings.dtype)
print(
    "Same values:",
    np.allclose(quran_embeddings, loaded_embeddings)
)

Loaded shape: (6236, 384)
Loaded dtype: float32
Same values: True


<div style="direction: rtl; white-space: normal; line-height: 1;">
Notebook 04 کامل شد
</div>